## 下载数据集

In [44]:
import datasets
raw_dataset = datasets.load_dataset('Jiayi-Pan/Countdown-Tasks-3to4',split='train',cache_dir=r"E:\TinyZero\examples\data_preprocess\raw_data")
print(f"原始数据：{raw_dataset}")
print(f"原始数据的长度为：{len(raw_dataset)}")
print(f"原始数据的类型为：{type(raw_dataset)}")
print(f"原始数据的特征为：{raw_dataset.features}")
print(f"原始数据的前5个元素为：{raw_dataset[:5]}")
print(f"原始数据的列名为：{raw_dataset.column_names}")

Generating train split: 100%|██████████| 490364/490364 [00:00<00:00, 7560221.90 examples/s]

原始数据：Dataset({
    features: ['target', 'nums'],
    num_rows: 490364
})
原始数据的长度为：490364
原始数据的类型为：<class 'datasets.arrow_dataset.Dataset'>
原始数据的特征为：{'target': Value(dtype='int64', id=None), 'nums': Sequence(feature=Value(dtype='int64', id=None), length=-1, id=None)}
原始数据的前5个元素为：{'target': [98, 64, 28, 48, 17], 'nums': [[44, 19, 35], [63, 95, 96], [95, 11, 56], [19, 74, 45], [49, 41, 73]]}
原始数据的列名为：['target', 'nums']


## 查看数据集
- 数据集本身可以看作一个列表，列表中的每一个元素都是一个字典

In [45]:
raw_dataset[0]

{'target': 98, 'nums': [44, 19, 35]}

In [46]:
raw_dataset[0]["target"]

98

## 对数据集进行划分

In [47]:
TEST_SIZE = 1024
TRAIN_SIZE = 327680

assert len(raw_dataset) > TRAIN_SIZE + TEST_SIZE
train_dataset = raw_dataset.select(range(TRAIN_SIZE))
test_dataset = raw_dataset.select(range(TRAIN_SIZE, TRAIN_SIZE + TEST_SIZE))

In [48]:
train_dataset[0]

{'target': 98, 'nums': [44, 19, 35]}

## 将数据集处理为预定义的格式

In [49]:
def make_prefix(dp, template_type):
    target = dp['target']
    numbers = dp['nums']
    # NOTE: also need to change reward_score/countdown.py
    if template_type == 'base':
        """This works for any base model"""
        prefix = f"""A conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer.
User: Using the numbers {numbers}, create an equation that equals {target}. You can use basic arithmetic operations (+, -, *, /) and each number can only be used once. Show your work in <think> </think> tags. And return the final answer in <answer> </answer> tags, for example <answer> (1 + 2) / 3 </answer>.
Assistant: Let me solve this step by step.
<think>"""
    elif template_type == 'qwen-instruct':
        """This works for Qwen Instruct Models"""
        prefix = f"""<|im_start|>system\nYou are a helpful assistant. You first thinks about the reasoning process in the mind and then provides the user with the answer.<|im_end|>\n<|im_start|>user\n Using the numbers {numbers}, create an equation that equals {target}. You can use basic arithmetic operations (+, -, *, /) and each number can only be used once. Show your work in <think> </think> tags. And return the final answer in <answer> </answer> tags, for example <answer> (1 + 2) / 3 </answer>.<|im_end|>\n<|im_start|>assistant\nLet me solve this step by step.\n<think>"""
    return prefix

def make_map_fn(split):
    def process_fn(example, idx):
        question = make_prefix(example, template_type="qwen-instruct")
        solution = {
            "target": example['target'],
            "numbers": example['nums']
        }
        data = {
            "data_source": 'countdown',
            "prompt": [{
                "role": "user",
                "content": question,
            }],
            "ability": "math",
            "reward_model": {
                "style": "rule",
                "ground_truth": solution
            },
            "extra_info": {
                'split': split,
                'index': idx,
            }
        }
        return data
    return process_fn

In [50]:
train_dataset = train_dataset.map(make_map_fn("train"), with_indices=True)
train_dataset[0]

Map: 100%|██████████| 327680/327680 [00:16<00:00, 20095.57 examples/s]


{'target': 98,
 'nums': [44, 19, 35],
 'data_source': 'countdown',
 'prompt': [{'content': '<|im_start|>system\nYou are a helpful assistant. You first thinks about the reasoning process in the mind and then provides the user with the answer.<|im_end|>\n<|im_start|>user\n Using the numbers [44, 19, 35], create an equation that equals 98. You can use basic arithmetic operations (+, -, *, /) and each number can only be used once. Show your work in <think> </think> tags. And return the final answer in <answer> </answer> tags, for example <answer> (1 + 2) / 3 </answer>.<|im_end|>\n<|im_start|>assistant\nLet me solve this step by step.\n<think>',
   'role': 'user'}],
 'ability': 'math',
 'reward_model': {'ground_truth': {'numbers': [44, 19, 35], 'target': 98},
  'style': 'rule'},
 'extra_info': {'index': 0, 'split': 'train'}}

In [51]:
test_dataset = test_dataset.map(make_map_fn("train"), with_indices=True)
test_dataset[0]

Map: 100%|██████████| 1024/1024 [00:00<00:00, 18770.97 examples/s]


{'target': 36,
 'nums': [79, 17, 60],
 'data_source': 'countdown',
 'prompt': [{'content': '<|im_start|>system\nYou are a helpful assistant. You first thinks about the reasoning process in the mind and then provides the user with the answer.<|im_end|>\n<|im_start|>user\n Using the numbers [79, 17, 60], create an equation that equals 36. You can use basic arithmetic operations (+, -, *, /) and each number can only be used once. Show your work in <think> </think> tags. And return the final answer in <answer> </answer> tags, for example <answer> (1 + 2) / 3 </answer>.<|im_end|>\n<|im_start|>assistant\nLet me solve this step by step.\n<think>',
   'role': 'user'}],
 'ability': 'math',
 'reward_model': {'ground_truth': {'numbers': [79, 17, 60], 'target': 36},
  'style': 'rule'},
 'extra_info': {'index': 0, 'split': 'train'}}

In [52]:
import os
local_dir = r"E:\TinyZero\data"
train_dataset.to_parquet(os.path.join(local_dir, 'train.parquet'))
test_dataset.to_parquet(os.path.join(local_dir, 'test.parquet'))

Creating parquet from Arrow format: 100%|██████████| 2/2 [00:00<00:00, 500.30ba/s]


708544